# TP Final NO revisado — Misma CNN 1D pero entrenada/validada/testeada en GiantSteps (real)

**Materia:** Redes Neuronales y Deep Learning · Maestría en IA · Universidad de Palermo

**Diferencia con el revisado:** misma `KeyCNN1D` (80k params, onda cruda 16kHz 4s -> 24 clases), mismo baseline K-S, misma regla cross-harp `+5`.
Lo único que cambia es el **dataset**: NO hay sintético. Se entrena, valida y testea con **GiantSteps Key** (604 previews 2 min EDM, 24 keys, Knees et al. ISMIR 2015), split por **tema** 70/15/15 para no filtrar ventanas del mismo tema. Agregado: salida con **3 riffs** (fácil 2 notas / intermedio / recomendado).

**Uso en Colab:** `Subir cuaderno` → `GPU T4` → `Reiniciar y ejecutar todo`. La descarga de ~604 MP3 tarda varios minutos la primera vez.


In [ ]:
# Celda 0 — Entorno + reproducibilidad
import sys, platform, random  # importa datos del sistema y azar base
import numpy as np  # importa calculo con vectores y azar
print("Python", sys.version.split()[0], "|", platform.system())  # muestra version de Python y sistema operativo
for n in ["numpy","matplotlib","pandas","sklearn","torch","torchaudio","librosa","scipy","seaborn"]:  # recorre las librerias a verificar
    try:  # intenta importar la libreria de turno
        m=__import__(n); print(f"{n:12s} {getattr(m,'__version__','ok')}")  # importa por nombre y muestra su version
    except ImportError: print(f"{n:12s} NO INSTALADO")  # si falta avisa y sigue con la proxima
def fijar_semillas(seed=42):  # define como fijar semillas para repetir resultados
    random.seed(seed); np.random.seed(seed)  # fija el azar de Python y numpy
    try:  # intenta configurar las semillas de torch
        import torch; torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)  # fija el azar de torch en CPU y GPU
        torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False  # fuerza calculo deterministico en GPU
    except ImportError: pass  # si falta el modulo sigue sin cortar
fijar_semillas(42); print("\nSemilla 42 OK")  # aplica semilla 42 y lo confirma
try:  # intenta detectar la GPU disponible
    import torch  # importa torch para consultar el hardware
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (alcanza)")  # muestra GPU o avisa que alcanza con CPU
except ImportError: pass  # si falta el modulo sigue sin cortar


## PASO 1 — Teoría acotada (24 keys, 4ta justa, CNN 1D vs K-S)

**Tonalidad (24 clases):** 12 tónicas x {mayor, menor}. No es la nota aislada: es el centro tonal + modo. Ej `A mayor` vs `A menor` comparten tónica pero distinta 3ra (C# vs C).

**Regla 2da posición (Cross-Harp):** `harm = (tonica_cancion + 5) mod 12`. Una 4ta justa arriba. Ej `G(7)->(7+5)%12=0->C`. Vale para mayor y menor (depende solo de tónica).

**CNN 1D:** kernel ancho 128 (~8 ms @16kHz) aprende periodicidad sin FFT; `stride/pool` dan invariancia temporal; `AdaptiveAvgPool1d(1)` colapsa tiempo. Entrada `(1, 64000)` 4 s → 24 logits.

**Baseline K-S (sin redes):** cromagrama de 12 bins promediado en tiempo `c` se correlaciona (Pearson) con 24 perfiles rotados: `MAJ=[6.35,2.23,...]`, `MIN=[6.33,2.68,...]` (Krumhansl-Kessler). Gana el key con mayor `r`. Si la CNN no supera esto, no agregó valor.


## PASO 2 — Dependencias

In [ ]:
# instala faltantes en Colab en silencio
!pip install -q librosa seaborn scikit-learn scipy 2>&1 | tail -n 2
import os, math, random  # importa sistema, matematica y azar
import numpy as np, matplotlib.pyplot as plt, seaborn as sns  # importa numeros, graficos y estilo de graficos
from scipy.io import wavfile  # importa lectura y escritura de WAV
import torch, torch.nn as nn  # importa torch y sus capas de red
from torch.utils.data import Dataset, DataLoader  # importa base de dataset y cargador por lotes
import torchaudio, librosa  # importa audio de torch y analisis de audio
from sklearn.metrics import confusion_matrix, accuracy_score  # importa matriz de confusion y accuracy
# muestra los graficos dentro del notebook
%matplotlib inline
sns.set(style="whitegrid")  # usa estilo con grilla en los graficos
DEVICE="cuda" if torch.cuda.is_available() else "cpu"  # elige GPU si hay y si no CPU
print("DEVICE:",DEVICE,"| torch:",torch.__version__)  # muestra dispositivo y version de torch


## PASO 3 — Dataset GiantSteps Key (real, 24 clases, split por tema)

Cada sample = 4 s @16kHz = 64000 floats normalizados por pico. Cada tema aporta hasta 8 ventanas (repartidas, sin primer/último tramo).
Split **por tema** 70/15/15 estratificado cuando hay muestras suficientes, si no aleatorio con semilla 7. Así ninguna ventana del mismo tema aparece en dos particiones.
Etiqueta: anotación `.key` normalizada (`Bb major` -> `A# maj`). La red NUNCA vio el test.


In [ ]:
SR=16000; DUR=4.0; N_SAMPLES=int(SR*DUR)  # 16kHz, 4s, 64000 muestras
TONICS=["C","C#","D","D#","E","F","F#","G","G#","A","A#","B"]  # 12 tonicas
T2I={n:i for i,n in enumerate(TONICS)}
KEYS=[f"{t} maj" for t in TONICS]+[f"{t} min" for t in TONICS]  # 24 clases
K2I={k:i for i,k in enumerate(KEYS)}
print("24 clases:",KEYS)
WINDOWS_PER_SONG=8; MAX_SONGS=None  # None = los 604; baja a 120 para prueba rapida

import subprocess, hashlib, urllib.request
GS_REPO="https://github.com/GiantSteps/giantsteps-key-dataset.git"
GS_DIR="/tmp/gs_key"; GS_MP3="https://www.cp.jku.at/datasets/giantsteps/backup/"
FLAT={"Db":"C#","Eb":"D#","Gb":"F#","Ab":"G#","Bb":"A#"}
def norm_key(txt):
    t,m=txt.strip().split(); return f"{FLAT.get(t,t)} {'maj' if m=='major' else 'min'}"

def descargar_todo_gs():
    if not os.path.isdir(os.path.join(GS_DIR,"annotations","key")):
        print("Clonando anotaciones GiantSteps..."); subprocess.run(["git","clone","-q","--depth","1",GS_REPO,GS_DIR],check=True)
    ids=sorted(f[:-4] for f in os.listdir(os.path.join(GS_DIR,"annotations","key")) if f.endswith(".key"))
    os.makedirs("/tmp/gs_audio",exist_ok=True); items=[]
    for i in ids:
        try:
            key=norm_key(open(os.path.join(GS_DIR,"annotations","key",i+".key")).read())
            if key not in K2I: print(f"x {i}: key {key} fuera de 24"); continue
            mp3=f"/tmp/gs_audio/{i}.mp3"
            if not (os.path.exists(mp3) and os.path.getsize(mp3)>50000):
                urllib.request.urlretrieve(GS_MP3+i+".mp3",mp3)
            if os.path.getsize(mp3)<50000: print(f"x {i}: descarga incompleta"); continue
            ref=open(os.path.join(GS_DIR,"md5",i+".md5")).read().strip()
            if hashlib.md5(open(mp3,"rb").read()).hexdigest()!=ref: print(f"x {i}: md5 distinto"); continue
            items.append((mp3,key))
        except Exception as e: print(f"x {i}: {e}")
    print(f"GiantSteps: {len(items)} audios OK"); return items

ITEMS=descargar_todo_gs()
if MAX_SONGS: ITEMS=random.Random(7).sample(ITEMS,min(MAX_SONGS,len(ITEMS))); print(f"Subconjunto: {len(ITEMS)} temas")

def ventanas_tema(path,n=WINDOWS_PER_SONG):
    y,_=librosa.load(path,sr=SR,mono=True); total=int(SR*DUR)
    if len(y)<total: y=np.pad(y,(0,total-len(y)))
    idxs=np.linspace(0,max(len(y)-total,0),min(n,max(len(y)//total,1))).round().astype(int) if len(y)>=total else [0]
    vs=[]
    for s in (idxs if len(y)>=total else [0]):
        s=int(s); w=y[s:s+total]
        if len(w)<total: w=np.pad(w,(0,total-len(w)))
        if np.max(np.abs(w))<1e-4: continue
        vs.append((0.95*w/(np.max(np.abs(w))+1e-9)).astype(np.float32))
        if len(vs)>=n: break
    return vs

# split por tema (sin fuga de ventanas)
from collections import Counter
labs=[k for _,k in ITEMS]
try:
    from sklearn.model_selection import train_test_split
    tr_ids,te_ids,_,_=train_test_split(ITEMS,labs,test_size=0.30,random_state=7,stratify=labs)
    tr_ids,va_ids=train_test_split(tr_ids,test_size=3/7,random_state=7)  # 70/15/15 aprox
except Exception as e:
    print("stratify fallo, split aleatorio:",e); r=random.Random(7); ids=ITEMS[:]; r.shuffle(ids)
    a,b=int(.7*len(ids)),int(.85*len(ids)); tr_ids,va_ids,te_ids=ids[:a],ids[a:b],ids[b:]
print(f"Temas train {len(tr_ids)} val {len(va_ids)} test {len(te_ids)} | dist train:",dict(Counter(labs).most_common(4)),"...")

def materializar(ids):
    X,Y,S=[],[],[]
    for path,key in ids:
        for w in ventanas_tema(path):
            X.append(w); Y.append(K2I[key]); S.append(os.path.basename(path))
    return np.stack(X).astype(np.float32) if X else np.zeros((0,N_SAMPLES),np.float32),np.array(Y,np.int64),S
Xtr,Ytr,Str=materializar(tr_ids); Xva,Yva,Sva=materializar(va_ids); Xte,Yte,Ste=materializar(te_ids)
print(f"Ventanas train {len(Ytr)} val {len(Yva)} test {len(Yte)}")
print("Test por clase (top):",dict(Counter([KEYS[i] for i in Yte]).most_common(5)))

class KeyDataset(Dataset):
    def __init__(self,X,Y): self.X=X; self.Y=Y
    def __len__(self): return len(self.Y)
    def __getitem__(self,idx): return torch.from_numpy(self.X[idx]).unsqueeze(0),torch.tensor(int(self.Y[idx]),dtype=torch.long)
BATCH=32
train_ds, val_ds, test_ds = KeyDataset(Xtr,Ytr), KeyDataset(Xva,Yva), KeyDataset(Xte,Yte)
train_loader=DataLoader(train_ds,batch_size=BATCH,shuffle=True)
val_loader=DataLoader(val_ds,batch_size=BATCH); test_loader=DataLoader(test_ds,batch_size=BATCH)
w0,y0=val_ds[0]; print("ej:",KEYS[y0.item()],tuple(w0.shape))
plt.figure(figsize=(10,2.5)); plt.plot(w0.numpy()[0,:8000]); plt.title(f"GiantSteps real 4s (zoom) - {KEYS[y0.item()]}"); plt.show()


## PASO 4 — CNN 1D a 24 clases

In [ ]:
class KeyCNN1D(nn.Module):  # red CNN 1D de 24 salidas
    def __init__(self,n_classes=24,dropout=0.3):  # guarda cantidad de clases y dropout
        super().__init__()  # inicializa la base de PyTorch
        self.b1=nn.Sequential(nn.Conv1d(1,16,kernel_size=128,stride=8,padding=64),nn.BatchNorm1d(16),nn.ReLU(),nn.MaxPool1d(4))  # bloque 1 con kernel ancho que capta graves y comprime
        self.b2=nn.Sequential(nn.Conv1d(16,32,kernel_size=16,padding=8),nn.BatchNorm1d(32),nn.ReLU(),nn.MaxPool1d(4))  # bloque 2 con kernel 16 para armonicos y comprime
        self.b3=nn.Sequential(nn.Conv1d(32,64,kernel_size=7,padding=3),nn.BatchNorm1d(64),nn.ReLU(),nn.MaxPool1d(4))  # bloque 3 con kernel 7 que afina timbre y comprime
        self.b4=nn.Sequential(nn.Conv1d(64,128,kernel_size=5,padding=2),nn.BatchNorm1d(128),nn.ReLU(),nn.MaxPool1d(4))  # bloque 4 con kernel 5 hasta 128 canales y comprime
        self.gap=nn.AdaptiveAvgPool1d(1)  # promedia todo el tiempo en un vector fijo
        self.head=nn.Sequential(nn.Flatten(),nn.Linear(128,96),nn.ReLU(),nn.Dropout(dropout),nn.Linear(96,n_classes))  # clasificador de 128 a 96 a 24 con dropout
    def forward(self,x):  # define la pasada hacia adelante de la red
        x=self.b1(x); x=self.b2(x); x=self.b3(x); x=self.b4(x); return self.head(self.gap(x))  # aplica bloques, promedia tiempo y clasifica
model=KeyCNN1D(24).to(DEVICE)  # crea el modelo en GPU o CPU
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")  # muestra cantidad de parametros
with torch.no_grad(): print("in",tuple(torch.randn(2,1,N_SAMPLES).to(DEVICE).shape),"-> out",tuple(model(torch.randn(2,1,N_SAMPLES).to(DEVICE)).shape))  # prueba formas entrada salida sin gradientes


## PASO 5 — Entrenamiento (CrossEntropy 24 + Adam)

In [ ]:
EPOCHS=12; LR=1e-3  # 12 epocas y paso de aprendizaje chico y seguro
crit=nn.CrossEntropyLoss(); opt=torch.optim.Adam(model.parameters(),lr=LR)  # perdida multiclase y optimizador Adam
def epoch(loader,train):  # corre una epoca de train o de evaluacion
    model.train(train)  # activa modo entrenar o modo evaluar
    tl,ok,n=0.0,0,0  # acumuladores de perdida, aciertos y total
    for xb,yb in loader:  # recorre los lotes del cargador
        xb,yb=xb.to(DEVICE),yb.to(DEVICE)  # manda el lote al dispositivo
        if train: opt.zero_grad()  # borra gradientes anteriores al entrenar
        with torch.set_grad_enabled(train):  # usa gradientes solo al entrenar
            lg=model(xb); loss=crit(lg,yb)  # predice el lote y calcula la perdida
            if train: loss.backward(); opt.step()  # retropropaga y actualiza pesos
        tl+=loss.item()*len(xb); ok+=(lg.argmax(1)==yb).sum().item(); n+=len(xb)  # acumula perdida y aciertos del lote
    return tl/n, ok/n  # devuelve perdida y accuracy promedio
hist={"tl":[],"vl":[],"ta":[],"va":[]}; best,bs=0,None  # historial de curvas y mejor modelo
for ep in range(1,EPOCHS+1):  # repite las 12 epocas
    a,b=epoch(train_loader,True); c,d=epoch(val_loader,False)  # entrena una epoca y valida
    hist["tl"].append(a); hist["vl"].append(c); hist["ta"].append(b); hist["va"].append(d)  # guarda metricas para graficar
    if d>best: best=d; bs={k:v.cpu().clone() for k,v in model.state_dict().items()}  # guarda copia en CPU del mejor modelo
    print(f"Ep {ep:02d} train {a:.4f}/{b:.3f} || val {c:.4f}/{d:.3f}")  # muestra el progreso de la epoca
model.load_state_dict({k:v.to(DEVICE) for k,v in bs.items()}); print(f"Best val {best:.3f}")  # restaura el mejor y muestra su accuracy


## PASO 6 — Evaluación: curvas + matriz 24x24 + score MIREX

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4))  # figura con dos graficos lado a lado
ax[0].plot(hist["tl"],label="train"); ax[0].plot(hist["vl"],label="val"); ax[0].set_title("Loss"); ax[0].legend()  # curva de perdida de train contra val
ax[1].plot(hist["ta"],label="train"); ax[1].plot(hist["va"],label="val"); ax[1].set_title("Acc"); ax[1].legend(); plt.show()  # curva de accuracy de train contra val
model.eval(); P,T=[],[]  # modo evaluacion y listas de prediccion y verdad
with torch.no_grad():  # evalua sin calcular gradientes
    for xb,yb in test_loader: P+=model(xb.to(DEVICE)).argmax(1).cpu().tolist(); T+=yb.tolist()  # junta predicciones y etiquetas de todo test
print(f"Test acc: {accuracy_score(T,P):.3f}")  # muestra accuracy final en test
cm=confusion_matrix(T,P,labels=list(range(24)))  # matriz de confusion de 24 por 24
plt.figure(figsize=(11,9)); sns.heatmap(cm,annot=False,cmap="Blues",xticklabels=KEYS,yticklabels=KEYS)  # dibuja la matriz como mapa de calor
plt.title("Confusion 24x24 (test sintetico)"); plt.xticks(rotation=90); plt.tight_layout(); plt.show()  # titula, rota etiquetas y muestra
# Score estilo MIREX: correcta=1, quinta=0.5, relativa/paralela=0.3, otro=0
def mirex(t,p):  # puntaje estilo MIREX con credito parcial
    if t==p: return 1.0  # acierto exacto vale 1 punto
    tt,mt=t%12,t<12; pt,mp=p%12,p<12  # separa tonica y modo de verdad y prediccion
    if mt==mp and (pt-tt)%12==7: return 0.5  # quinta del mismo modo vale 0 punto 5
    if (mt!=mp) and ((tt==0 and pt==9) or (pt==0 and tt==9) or ((pt-tt)%12==9 and mt) or ((tt-pt)%12==9 and mp)): return 0.3  # relativa o cambio de modo vale 0 punto 3
    if tt==pt and mt!=mp: return 0.3  # misma tonica en otro modo vale 0 punto 3
    return 0.0  # cualquier otro error vale 0
print(f"MIREX ponderado: {np.mean([mirex(t,p) for t,p in zip(T,P)]):.3f}")  # muestra promedio MIREX en test


## PASO 7 — Regla armónica (4ta justa, +5 semitonos)

Solo 3 acciones: soplar / aspirar / bending (aspirar frenando el aire, suena lloradito). La tabla no se memoriza: sale de la fórmula.

In [ ]:
def cuarta_arriba(tonica): return TONICS[(T2I[tonica]+5)%12]  # 2da posicion
CROSS={t:cuarta_arriba(t) for t in TONICS}  # tabla cancion a armonica con la regla mas 5
print("Cross-Harp (regla +5):",CROSS)  # muestra la tabla completa
# Riff detallado para C; el resto usa el riff generico de 2da posicion (celdas 2-4, casa en 2 aspirando).
RIFF_D={"C":["celda 2 aspirando","celda 3 aspirando con bending","celda 4 aspirando","celda 4 soplando","celda 4 aspirando","celda 3 aspirando con bending","celda 2 aspirando"],"default":["celda 2 aspirando (casa)","celda 3 aspirando con bending (lloradito)","celda 4 aspirando","celda 4 soplando","celda 4 aspirando","celda 3 aspirando con bending","celda 2 aspirando"]}
def recomendar(tonica):  # devuelve armonica y riff para una tonica
    h=CROSS[tonica]; return h, RIFF_D.get(h,RIFF_D["default"])  # busca armonica y riff o usa el generico
for d in ["G","A","E"]: print(d,"-> armonica",recomendar(d)[0])  # demo rapida con tres ejemplos


## PASO 8 — Audio propio para demo (el entrenamiento ya usó GiantSteps)

**Entrada:** `WAV/MP3` -> mono 16kHz -> ventanas de 4 s. El dataset GiantSteps ya se usó en PASO 3 (train/val/test por tema).
Acá solo se arma una demo + upload manual para probar la cadena completa en un tema fuera del test.


In [ ]:
def cargar_audio_Xs(path,sr=SR,dur=DUR):  # carga un audio y lo corta en ventanas
    y,_=librosa.load(path,sr=sr,mono=True); n=int(sr*dur); vs=[]  # lee mono a 16kHz y prepara lista de ventanas
    for i in range(0,max(len(y),1),n):  # recorre el audio de a ventanas
        s=y[i:i+n];  # toma el tramo actual
        s=np.pad(s,(0,n-len(s))) if len(s)<n else s  # rellena cola corta
        vs.append((0.95*s/(np.max(np.abs(s))+1e-9)).astype(np.float32))  # normaliza por pico
        if i+n>=len(y): break
    print(f"{os.path.basename(path)}: {len(y)/sr:.1f}s -> {len(vs)} ventanas {dur}s"); return vs
# Demo propia fuera del test: 12-bar blues en A (etiqueta A maj para este TP)
def _f(m): return 440.0*(2.0**((m-69)/12.0))  # MIDI a Hz
def _n(fr,d,sr=SR,rng=None,g=0.4):  # nota corta con 3 armonicos
    rng=np.random if rng is None else rng; n=int(sr*d); t=np.arange(n,dtype=np.float32)/sr; x=np.zeros(n,dtype=np.float32)
    for h,a in enumerate([1.0,0.45,0.22],start=1): x+=(a*np.sin(2*np.pi*fr*h*t+rng.uniform(0,6.28))).astype(np.float32)
    return (g*x).astype(np.float32)
def demo_blues(key="A",sr=SR,seed=7):  # blues 12 compases
    rng=np.random.RandomState(seed); BAR=2.0; roots=[45,45,50,45,52,50,45,52,45,50,45,52] if key=="A" else [43,43,48,43,50,48,43,50,43,48,43,50]
    out=np.zeros(int(sr*BAR*12),dtype=np.float32)
    for b,r in enumerate(roots):
        base=b*int(sr*BAR); bse=_f(r-12)
        out[base:base+int(sr*2.0)]+=np.concatenate([_n(bse,0.5,sr,rng,0.5) for _ in range(4)])[:int(sr*2.0)]
    out=np.tanh(out); return (0.9*out/(np.max(np.abs(out))+1e-9)).astype(np.float32)
os.makedirs("/tmp",exist_ok=True); DEMO="/tmp/demo_blues_A.wav"
wavfile.write(DEMO,SR,(demo_blues()*32767).astype(np.int16)); print("Demo:",DEMO,"| key verdadera: A maj")
MI_AUDIO=DEMO  # usa la demo salvo que se suba otro audio
try:
    from google.colab import files as _F; print("Subi MP3/WAV o cancela para demo:"); _u=_F.upload()
    if _u:
        _k=sorted(_u.keys())[0]; open(f"/content/{_k}","wb").write(_u[_k]); MI_AUDIO=f"/content/{_k}"
except ImportError: print("Local -> demo")
print("MI_AUDIO:",MI_AUDIO)


## PASO 9 — Inferencia CNN vs Baseline K-S

In [ ]:
MAJ=np.array([6.35,2.23,3.48,2.33,4.38,4.09,2.52,5.19,2.39,3.66,2.29,2.88],dtype=float)  # perfil mayor de Krumhansl y Kessler
MIN=np.array([6.33,2.68,3.52,5.38,2.60,3.53,2.54,4.75,3.98,2.69,3.34,3.17],dtype=float)  # perfil menor de Krumhansl y Kessler
def baseline_ks(path):  # estima la clave sin redes por cromagrama
    y,_=librosa.load(path,sr=SR,mono=True)  # lee el audio en mono a 16kHz
    ch=librosa.feature.chroma_cqt(y=y,sr=SR).mean(axis=1)  # cromagrama de 12 bandas promediado en tiempo
    best,bi=-2,0  # mejor correlacion y mejor clave hasta ahora
    for t in range(12):  # prueba las 12 tonicas
        for m,prof in [("maj",MAJ),("min",MIN)]:  # prueba modo mayor y modo menor
            pr=np.roll(prof,t)  # rota el perfil hasta la tonica t
            r=np.corrcoef(ch,pr)[0,1]  # correlacion de Pearson entre croma y perfil
            if r>best: best,bi=r,(t,m)  # guarda la clave con mayor correlacion
    lab=f"{TONICS[bi[0]]} {bi[1]}"; return lab,float(best)  # arma la etiqueta y devuelve con su r
def predecir_cnn(path,model=model,maxv=6,verbose=True):  # predice la clave con la CNN por voto
    model.eval(); vs=cargar_audio_Xs(path)  # pone modo evaluar y corta el audio en ventanas
    
    if len(vs)>maxv: vs=[vs[i] for i in np.linspace(0,len(vs)-1,maxv).round().astype(int)]  # en audios largos muestrea 6 ventanas parejas
    xb=torch.stack([torch.from_numpy(w).unsqueeze(0) for w in vs]).to(DEVICE)  # arma el lote de ventanas en el dispositivo
    with torch.no_grad(): v=torch.softmax(model(xb),1).mean(0)  # promedia probabilidades sin gradientes
    i=int(v.argmax()); top=sorted(range(24),key=lambda k: float(v[k]),reverse=True)[:3]  # toma la ganadora y el top 3
    if verbose: print("CNN Top-3:",{KEYS[k]:round(float(v[k]),3) for k in top})  # muestra el top 3 solo si se pide
    return KEYS[i],float(v[i])  # devuelve etiqueta y confianza
cnn_lab,cnn_c=predecir_cnn(MI_AUDIO); ks_lab,ks_r=baseline_ks(MI_AUDIO)  # predice el audio con ambos metodos
print(f"\nCNN: {cnn_lab} ({cnn_c:.2f}) | K-S: {ks_lab} (r={ks_r:.2f}) | verdad demo: A maj")  # muestra CNN contra K-S y la verdad de la demo
ton_cnn=cnn_lab.split()[0]  # extrae la tonica para la armonica


## PASO 9b — Resultados en test GiantSteps holdout (temas nunca vistos)

Cada ventana del test se predice con la CNN. Además se vota por tema (promedio de probs de sus ventanas) para CNN y K-S. Se reporta accuracy ventana, accuracy tema y MIREX.


In [ ]:
# Evalua holdout a nivel tema: CNN por voto de ventanas + K-S por tema
print(f"{'tema':28s} {'verdad':8s} {'cnn':8s} {'ks':8s} cnn_ok ks_ok")
ok_c=ok_k=0; mc=mk=0.0; rows=[]
import torch as _t
for path,true in te_ids:
    vs=ventanas_tema(path,n=6)  # hasta 6 ventanas del tema
    xb=_t.stack([_t.from_numpy(w).unsqueeze(0) for w in vs]).to(DEVICE)
    with _t.no_grad(): v=_t.softmax(model(xb),1).mean(0)
    cnn=KEYS[int(v.argmax())]
    ks,_=baseline_ks(path)  # K-S sobre el tema completo
    cc,kk=cnn==true,ks==true; ok_c+=cc; ok_k+=kk
    sc,sk=mirex(K2I[true],K2I[cnn]),mirex(K2I[true],K2I[ks]); mc+=sc; mk+=sk; rows.append((path,true,cnn,ks))
    print(f"{os.path.basename(path):28s} {true:8s} {cnn:8s} {ks:8s} {int(cc):>6d} {int(kk):>5d}")
n=len(te_ids)
print(f"\nGiantSteps-TEST temas {n} | CNN acc {ok_c/n:.3f} mirex {mc/n:.3f} || K-S acc {ok_k/n:.3f} mirex {mk/n:.3f}")
print(f"(Ventanas test ya reportadas en PASO 6: acc {accuracy_score(T,P):.3f})")


## PASO 10 — As de la Armónica

In [ ]:
h,riff_pro=recomendar(ton_cnn)  # pide armonica y riff actual para la tonica
riff_facil=["celda 2 aspirando (casa)","celda 4 soplando"]  # super facil: 2 notas, solo aspirar/soplar
riff_medio=["celda 2 aspirando (casa)","celda 3 aspirando con bending (lloradito)","celda 4 aspirando","celda 4 soplando"]  # intermedio: suma bending
print("="*52); print("   EL AS DE LA ARMONICA"); print("="*52)  # dibuja el marco del informe final
print(f"Tonalidad CNN : {cnn_lab} (conf {cnn_c:.2f})")  # muestra tonalidad y confianza del modelo
print(f"Baseline K-S  : {ks_lab} (r={ks_r:.2f})")  # muestra baseline y su correlacion
print(f"Agarra         : ARMONICA EN {h}  (cancion {ton_cnn} +5 semitonos)")  # indica la armonica con la regla mas 5
print(" [1] FACIL 2 notas:")  # super facil alterna celdas 2 y 4
for i,p in enumerate(riff_facil,1): print(f"   {i}. {p}")
print(" [2] INTERMEDIO:")  # suma bending en celda 3
for i,p in enumerate(riff_medio,1): print(f"   {i}. {p}")
print(" [3] RECOMENDADO:")  # fraseo completo actual de 7 pasos
for i,p in enumerate(riff_pro,1): print(f"   {i}. {p}")
print("Tip: casa = celda 2 aspirando."); print("="*52)  # consejo final y cierre


## PASO 11 — Qué esperar al entrenar en real (vs sintético)

> Este notebook entrena en GiantSteps, no en sintético. El 100 % del revisado NO aplica acá.
> Espera accuracy ventana 0.3-0.6 y tema 0.4-0.7 según split, con K-S como vara (suele dar 0.4-0.5 en GiantSteps-20).
> Si la CNN supera a K-S en el holdout por tema, hay generalización real; si colapsa a una clase (ej todo `A maj`), es el mismo shortcut pero ahora sobre EDM.
> Compara matriz 24x24, Top-3 y MIREX, no solo accuracy.


### Cierre — para planilla y repo

**Descripción breve:** Misma `KeyCNN1D` (onda cruda 16kHz 4s -> 24 clases, CrossEntropy+Adam 12ep) pero entrenada/validada/testeada en **GiantSteps Key** real con split por tema 70/15/15. Baseline K-S. Salida con **3 riffs** (fácil 2 notas / intermedio / recomendado) + regla cross-harp `+5`.

**Dataset:** GiantSteps Key (604 previews 2min, 24 keys, Knees et al. ISMIR2015, mirror JKU con md5). Sin sintético. `WINDOWS_PER_SONG=8`, `MAX_SONGS=None` (bajar a 120 para prueba rápida).

**Límites:** desbalance de clases en EDM, 4s no capta modulaciones, blues mezcla mayor/menor → reportar Top-3 + MIREX.
